###Download libraries

In [1]:
!pip install torch
!pip install transformers
!pip install langgraph
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install faiss-cpu
!pip install sentence-transformers

# Models
!pip install accelerate

!pip install bitsandbytes

!pip install beautifulsoup4

!pip install requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 11.4 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 12.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 10.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]32m5/6 [torch]]x]
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 11.4 MB/s  0:00:00 eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)


##Scrape the data

In [2]:

import json
import difflib
import requests
import time
import random
import re
from bs4 import BeautifulSoup, NavigableString, Tag
from urllib.parse import urlparse, unquote
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==========================================
# 1. CORE DATA MODEL
# ==========================================
class RecipeEntity:
    def __init__(self, dish_name: str):
        self.dish_name = dish_name
        self.cuisine_type = "South Asian"
        self.introductions = []
        self.ingredients = []
        self.instructions = []
        self.source_urls = set()

    def _is_duplicate(self, new_text: str, existing_list: list, threshold: float = 0.93) -> bool:
        if not new_text or not new_text.strip():
            return True

        clean_new = normalize_text(new_text)
        if len(clean_new) < 20:
            return True

        for existing_text in existing_list:
            clean_existing = normalize_text(existing_text)
            if difflib.SequenceMatcher(None, clean_new, clean_existing).ratio() > threshold:
                return True
        return False

    def add_introduction(self, text: str, url: str):
        if not self._is_duplicate(text, self.introductions):
            self.introductions.append(text.strip())
            self.source_urls.add(url)

    def add_ingredients(self, text: str, url: str):
        if not self._is_duplicate(text, self.ingredients):
            self.ingredients.append(text.strip())
            self.source_urls.add(url)

    def add_instructions(self, text: str, url: str):
        if not self._is_duplicate(text, self.instructions):
            self.instructions.append(text.strip())
            self.source_urls.add(url)


# ==========================================
# 2. HELPERS
# ==========================================
JUNK_URL_PATTERNS = [
    "action=edit", "action=history", "veaction=edit", "printable=yes",
    "oldid=", "mobileaction=", "Special:", "Main_Page", "WhatLinksHere",
    "RecentChangesLinked", "action=info", "redlink=1",
]

NON_RECIPE_TITLES = {
    "Table of Contents", "South Asian Cuisine", "South Asian cuisines",
    "Asian Cuisine", "Cuisines", "Recipes", "Ingredients", "Equipment",
    "Cooking Techniques", "Cuisine of India", "Cuisine of Pakistan",
    "Cuisine of Nepal", "Cuisine of Afghanistan",
}

STOP_SECTION_WORDS = [
    "reference", "references", "see also", "notes", "external links",
    "gallery", "related", "navigation", "further reading"
]

INGREDIENT_SECTION_WORDS = [
    "ingredient", "ingredients", "what you need", "you will need", "needs"
]

INSTRUCTION_SECTION_WORDS = [
    "instruction", "instructions", "method", "methods", "direction",
    "directions", "preparation", "procedure", "steps", "cooking", "methodology", "make"
]


def normalize_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_clean_text(node: Tag) -> str:
    text = node.get_text(" ", strip=True)
    return normalize_text(text)


def is_junk_url(url: str) -> bool:
    return any(p.lower() in url.lower() for p in JUNK_URL_PATTERNS)


def looks_like_recipe_url(url: str) -> bool:
    parsed = urlparse(url)
    path = unquote(parsed.path)

    if "en.wikibooks.org" not in parsed.netloc:
        return False

    if "/wiki/Cookbook:" not in path:
        return False

    return True


def clean_dish_name_from_url(url: str) -> str:
    parsed_url = urlparse(url)
    if parsed_url.fragment:
        name = unquote(parsed_url.fragment).replace("_", " ")
    else:
        raw_name = parsed_url.path.split("/")[-1]
        name = unquote(raw_name).replace("_", " ").replace("Cookbook:", "")
    return name.strip()


def is_non_recipe_title(title: str) -> bool:
    stripped = title.strip()
    if stripped in NON_RECIPE_TITLES:
        return True

    lowered = stripped.lower()
    if lowered.startswith("cuisine of "):
        return True

    return False


def dedupe_urls(urls: list[str]) -> list[str]:
    seen = set()
    final = []
    for url in urls:
        clean = url.strip()
        if not clean or clean in seen:
            continue
        seen.add(clean)
        final.append(clean)
    return final


# ==========================================
# 3. BASE SCRAPER
# ==========================================
class BaseScraper:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "SouthAsianRecipeBot/1.0 (Academic Research Project)",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
        })

        retries = Retry(
            total=4,
            backoff_factor=2,
            status_forcelist=[403, 429, 500, 502, 503, 504],
            allowed_methods=["GET"],
        )
        adapter = HTTPAdapter(max_retries=retries)
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def fetch_soup(self, url: str) -> BeautifulSoup:
        sleep_time = random.uniform(1.5, 3.0)
        print(f"Fetching: {url} | delay {sleep_time:.1f}s")
        time.sleep(sleep_time)

        response = self.session.get(url, timeout=20)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")


# ==========================================
# 4. SCRAPERS
# ==========================================
class WikipediaScraper(BaseScraper):
    def scrape(self, url: str, entity: RecipeEntity):
        soup = self.fetch_soup(url)
        parsed_url = urlparse(url)
        intro_paragraphs = []

        if parsed_url.fragment:
            heading_span = soup.find(id=parsed_url.fragment)
            if heading_span:
                parent_h = heading_span.parent
                for sibling in parent_h.next_siblings:
                    if isinstance(sibling, Tag) and sibling.name in ["h2", "h3"]:
                        break
                    if isinstance(sibling, Tag) and sibling.name == "p":
                        text = extract_clean_text(sibling)
                        if text:
                            intro_paragraphs.append(text)
        else:
            content = soup.find(id="mw-content-text")
            if content:
                parser_output = content.find(class_="mw-parser-output")
                parent = parser_output if parser_output else content
                for element in parent.children:
                    if isinstance(element, NavigableString):
                        continue
                    if element.name == "p":
                        text = extract_clean_text(element)
                        if text:
                            intro_paragraphs.append(text)
                    elif element.name in ["h2", "h3"]:
                        break

        if intro_paragraphs:
            entity.add_introduction(" ".join(intro_paragraphs), url)


class WikibooksScraper(BaseScraper):
    def scrape(self, url: str, entity: RecipeEntity):
        if is_junk_url(url):
            print("Skipping junk/system URL")
            return

        soup = self.fetch_soup(url)
        content = soup.find(id="mw-content-text")
        if not content:
            return

        parser_output = content.find(class_="mw-parser-output")
        parent = parser_output if parser_output else content

        intro_parts = []
        ingredient_parts = []
        instruction_parts = []

        current_section = "intro"

        # iterate over direct children in order
        for element in parent.children:
            if isinstance(element, NavigableString):
                continue
            if not isinstance(element, Tag):
                continue

            heading = self._extract_heading_text(element)
            if heading is not None:
                heading_lower = heading.lower()

                if any(word in heading_lower for word in STOP_SECTION_WORDS):
                    break
                elif any(word in heading_lower for word in INGREDIENT_SECTION_WORDS):
                    current_section = "ingredients"
                    continue
                elif any(word in heading_lower for word in INSTRUCTION_SECTION_WORDS):
                    current_section = "instructions"
                    continue
                else:
                    continue

            block_texts = self._extract_block_content(element)

            if not block_texts:
                continue

            if current_section == "intro":
                intro_parts.extend(block_texts)
            elif current_section == "ingredients":
                ingredient_parts.extend(block_texts)
            elif current_section == "instructions":
                instruction_parts.extend(block_texts)

        intro_text = "\n".join(intro_parts).strip()
        ingredient_text = "\n".join(ingredient_parts).strip()
        instruction_text = "\n".join(instruction_parts).strip()

        # fallback heuristic: if nothing detected, try page-wide list extraction
        if not ingredient_text and not instruction_text:
            fallback_ing, fallback_inst = self._fallback_extract(parent)
            if fallback_ing:
                ingredient_text = fallback_ing
            if fallback_inst:
                instruction_text = fallback_inst

        if intro_text:
            entity.add_introduction(intro_text, url)
        if ingredient_text:
            entity.add_ingredients(ingredient_text, url)
        if instruction_text:
            entity.add_instructions(instruction_text, url)

    def _extract_heading_text(self, element: Tag):
        if element.name in ["h2", "h3", "h4", "h5"]:
            return extract_clean_text(element).replace("[ edit ]", "").replace("[edit]", "")

        if element.name == "div" and element.has_attr("class"):
            classes = " ".join(element.get("class", []))
            if "mw-heading" in classes:
                inner = element.find(["h2", "h3", "h4", "h5"])
                if inner:
                    return extract_clean_text(inner).replace("[ edit ]", "").replace("[edit]", "")

        return None

    def _extract_block_content(self, element: Tag) -> list[str]:
        output = []

        # Paragraphs
        if element.name == "p":
            text = extract_clean_text(element)
            if text:
                output.append(text)
            return output

        # Unordered or ordered lists
        if element.name in ["ul", "ol"]:
            for i, li in enumerate(element.find_all("li", recursive=False), start=1):
                li_text = extract_clean_text(li)
                if not li_text:
                    continue
                if element.name == "ol":
                    output.append(f"{i}. {li_text}")
                else:
                    output.append(f"- {li_text}")
            return output

        # Tables sometimes hold recipe content
        if element.name == "table":
            rows = element.find_all("tr")
            temp = []
            for row in rows:
                cells = row.find_all(["td", "th"])
                row_text = " | ".join(extract_clean_text(c) for c in cells if extract_clean_text(c))
                if row_text:
                    temp.append(row_text)
            return temp

        # Some pages wrap content in divs
        if element.name == "div":
            paragraphs = element.find_all("p", recursive=False)
            lists = element.find_all(["ul", "ol"], recursive=False)

            for p in paragraphs:
                text = extract_clean_text(p)
                if text:
                    output.append(text)

            for lst in lists:
                for i, li in enumerate(lst.find_all("li", recursive=False), start=1):
                    li_text = extract_clean_text(li)
                    if not li_text:
                        continue
                    if lst.name == "ol":
                        output.append(f"{i}. {li_text}")
                    else:
                        output.append(f"- {li_text}")

            return output

        return output

    def _fallback_extract(self, parent: Tag):
        all_lists = parent.find_all(["ul", "ol"])
        ingredient_lines = []
        instruction_lines = []

        for lst in all_lists:
            items = [extract_clean_text(li) for li in lst.find_all("li")]
            items = [x for x in items if x]
            if not items:
                continue

            # rough heuristic
            numbered_like = 0
            ingredient_like = 0

            for item in items:
                if re.search(r"\b(cup|cups|tbsp|tsp|teaspoon|tablespoon|gram|grams|kg|ml|litre|liter)\b", item.lower()):
                    ingredient_like += 1
                if re.search(r"\b(stir|add|mix|heat|boil|cook|serve|fry|bake|simmer)\b", item.lower()):
                    numbered_like += 1

            if ingredient_like >= max(2, len(items) // 3):
                ingredient_lines.extend([f"- {x}" for x in items])
            elif numbered_like >= max(2, len(items) // 3):
                instruction_lines.extend([f"{i+1}. {x}" for i, x in enumerate(items)])

        return "\n".join(ingredient_lines).strip(), "\n".join(instruction_lines).strip()


class BlogScraper(BaseScraper):
    def scrape(self, url: str, entity: RecipeEntity):
        soup = self.fetch_soup(url)
        # placeholder
        pass


# ==========================================
# 5. ORCHESTRATOR
# ==========================================
class IngestionPipeline:
    def __init__(self):
        self.scrapers = {
            "en.wikipedia.org": WikipediaScraper(),
            "en.wikibooks.org": WikibooksScraper(),
            "aroundtheworldin80cuisinesblog.wordpress.com": BlogScraper(),
        }
        self.database = {}

    def extract_dish_name(self, url: str) -> str:
        return clean_dish_name_from_url(url)

    def should_process_url(self, url: str) -> bool:
        if is_junk_url(url):
            return False

        dish_name = self.extract_dish_name(url)
        if is_non_recipe_title(dish_name):
            return False

        parsed = urlparse(url)

        if parsed.netloc == "en.wikibooks.org":
            return looks_like_recipe_url(url)

        if parsed.netloc == "en.wikipedia.org":
            return "South_Asian" in url or "South_Asian_cuisine" in url

        return parsed.netloc in self.scrapers

    def process_urls(self, urls: list[str]):
        filtered_urls = dedupe_urls(urls)

        kept = 0
        skipped = 0

        for url in filtered_urls:
            if not self.should_process_url(url):
                print(f"Skipping non-recipe/junk URL: {url}")
                skipped += 1
                continue

            try:
                domain = urlparse(url).netloc
                dish_name = self.extract_dish_name(url)

                if dish_name not in self.database:
                    self.database[dish_name] = RecipeEntity(dish_name)
                entity = self.database[dish_name]

                scraper = self.scrapers.get(domain)
                if not scraper:
                    print(f"Warning: no scraper configured for domain {domain}")
                    skipped += 1
                    continue

                print(f"Scraping {domain} -> {dish_name}")
                scraper.scrape(url, entity)
                kept += 1

            except Exception as e:
                print(f"Error processing {url}: {e}")

        print(f"\nProcessed URLs: {kept} | Skipped URLs: {skipped}")

    # ==========================================
    # MODIFIED: Output a single document per dish
    # ==========================================
    def generate_consolidated_json(self) -> list:
        documents = []
        dish_counter = 1

        for dish_name, entity in sorted(self.database.items()):
            # Skip if we didn't extract any meaningful recipe content
            if not entity.introductions and not entity.ingredients and not entity.instructions:
                continue

            dish_id = f"wiki_southasian_{dish_counter:03d}"
            primary_url = next(iter(entity.source_urls), "")

            # Combine everything into a single text block
            full_text_parts = [f"Title: {dish_name}\n"]

            if entity.introductions:
                full_text_parts.append("--- Introduction ---")
                full_text_parts.extend(entity.introductions)

            if entity.ingredients:
                full_text_parts.append("\n--- Ingredients ---")
                full_text_parts.extend(entity.ingredients)

            if entity.instructions:
                full_text_parts.append("\n--- Instructions ---")
                full_text_parts.extend(entity.instructions)

            full_text = "\n".join(full_text_parts).strip()

            # Create ONE document containing the entire recipe
            documents.append({
                "id": dish_id,
                "source_url": primary_url,
                "title": dish_name,
                "cuisine_type": entity.cuisine_type,
                "full_text": full_text,
                # Blank template ready for the LLM enrichment script
                "metadata": {
                    "diet": "",
                    "prep_time": "",
                    "dish_type": ""
                }
            })
            dish_counter += 1

        return documents


# ==========================================
# 6. EXECUTION
# ==========================================
if __name__ == "__main__":
    target_links = [
        "https://en.wikibooks.org/wiki/Cookbook:Table_of_Contents",
        "https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine",
        "https://en.wikibooks.org/w/index.php?title=Cookbook:South_Asian_Cuisine&action=edit",
        "https://en.wikibooks.org/w/index.php?title=Cookbook:South_Asian_Cuisine&action=history",
        "https://en.wikibooks.org/wiki/Special:WhatLinksHere/Cookbook:South_Asian_Cuisine",
        "https://en.wikibooks.org/wiki/Special:RecentChangesLinked/Cookbook:South_Asian_Cuisine",
        "https://en.wikipedia.org/wiki/South_Asian_cuisine",
        "https://en.wikibooks.org/wiki/Cookbook:Corom_Chatni_(Mango_Chutney_with_Hot_Chillies)",
        "https://en.wikibooks.org/wiki/Cookbook:Dum_ka_Qimah_(Spiced_Minced_Meat)",
        "https://en.wikibooks.org/wiki/Cookbook:Khatti_Dal_(Spiced_Tamarind_Pigeon_Peas)",
        "https://en.wikibooks.org/wiki/Cookbook:Malpua_(South_Asian_Sweet_Pancake)",
        "https://en.wikibooks.org/wiki/Cookbook:Mango_Chutney_(Chunky)",
        "https://en.wikibooks.org/wiki/Cookbook:Mango_Chutney_(Smooth)",
        "https://en.wikibooks.org/wiki/Cookbook:Masala_Chai_II",
        "https://en.wikibooks.org/wiki/Cookbook:Masyaura_(Nepali_Fermented_Vegetable_Balls)",
        "https://en.wikibooks.org/wiki/Cookbook:Mild_Salty_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Papadam_(Black_Gram_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Papaya_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Papri_Chaat_(Crispy_Indian_Snack_with_Potato)",
        "https://en.wikibooks.org/wiki/Cookbook:Phulourie_(Split_Pea_Fritters)",
        "https://en.wikibooks.org/wiki/Cookbook:Prawn_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Qabuli_(Central_Asian_Rice_Pilaf)",
        "https://en.wikibooks.org/wiki/Cookbook:Seviyan_Ji_Khirni_(Sindhi_Vermicelli_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:Sindhi_Fried_Potatoes",
        "https://en.wikibooks.org/wiki/Cookbook:Sweet_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Sweet_Mango_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Doner_Kebab",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Fried_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Handesh",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Rice_Pudding",
        "https://en.wikibooks.org/wiki/Cookbook:Watalappam_(Sri_Lankan_Coconut_Custard)",
        "https://en.wikibooks.org/w/index.php?title=Cookbook:South_Asian_Cuisine&mobileaction=toggle_view_mobile",
        "https://en.wikibooks.org/wiki/Cookbook:Cuisine_of_Afghanistan",
        "https://en.wikibooks.org/wiki/Cookbook:Afghan_Bread",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Tikka",
        "https://en.wikibooks.org/wiki/Cookbook:Naan",
        "https://en.wikibooks.org/wiki/Cookbook:Recipes",
        "https://en.wikibooks.org/wiki/Cookbook:Ingredients",
        "https://en.wikibooks.org/wiki/Cookbook:Equipment",
        "https://en.wikibooks.org/wiki/Cookbook:Cooking_Techniques",
        "https://en.wikibooks.org/w/index.php?title=Cookbook:Cuisine_of_Bengal&action=edit&redlink=1",
        "https://en.wikibooks.org/wiki/Cookbook:Arisa_Pitha_(Fried_Indian_Sweet_Rice_Pastry)",
        "https://en.wikibooks.org/wiki/Cookbook:Chyapa_Shutki_Bharta",
        "https://en.wikibooks.org/wiki/Cookbook:Bhuna_Khichuri_(Bengali_Rice_and_Lentils)",
        "https://en.wikibooks.org/wiki/Cookbook:Mishti_Doi_(Bengali_Sweetened_Yogurt)",
        "https://en.wikibooks.org/wiki/Cookbook:Murghi_Korma_(Chicken_Korma)",
        "https://en.wikibooks.org/wiki/Cookbook:Pudina_Hilsa_(Bengali_Fish_with_Mint)",
        "https://en.wikibooks.org/wiki/Cookbook:Rosogulla_(Bengali_Milk_Balls_in_Syrup)",
        "https://en.wikibooks.org/wiki/Cookbook:Cuisine_of_India",
        "https://en.wikibooks.org/wiki/Cookbook:Potato_Curry_(Aloo_Masala)",
        "https://en.wikibooks.org/wiki/Cookbook:Baingan_Bartha_(South_Indian_Eggplant_with_Chili)_II",
        "https://en.wikibooks.org/wiki/Cookbook:Baingan_Bartha_(South_Indian_Eggplant_with_Coconut_and_Chili)_I",
        "https://en.wikibooks.org/wiki/Cookbook:Basic_Indian_Tomato_Gravy",
        "https://en.wikibooks.org/wiki/Cookbook:Bengal_Potatoes",
        "https://en.wikibooks.org/wiki/Cookbook:Bonda_(South_Indian_Vegetable_Fritter)",
        "https://en.wikibooks.org/wiki/Cookbook:Borhani_(Spiced_Yogurt_Drink)",
        "https://en.wikibooks.org/wiki/Cookbook:Bread_Filled_with_Potato_Curry_(Pani_Puri)",
        "https://en.wikibooks.org/wiki/Cookbook:Buttermilk_Curry_Soup_(Kadi_Pakora)",
        "https://en.wikibooks.org/wiki/Cookbook:Chakarai_Pongal_(Sweet_Rice_and_Black_Gram_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:Chapati",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Curry_(Mediterranean-inspired)",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Madras",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Tikka_Masala",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Vindaloo",
        "https://en.wikibooks.org/wiki/Cookbook:Chickpea_Curry_(Masaledaar_Chole)",
        "https://en.wikibooks.org/wiki/Cookbook:Cholley_(Chickpea_Curry)",
        "https://en.wikibooks.org/wiki/Cookbook:Churri_(Indian_Yogurt_Herb_Sauce)",
        "https://en.wikibooks.org/wiki/Cookbook:Coconut_Barfi",
        "https://en.wikibooks.org/wiki/Cookbook:Coconut_Chutney_(North_Indian)",
        "https://en.wikibooks.org/wiki/Cookbook:Coconut_Chutney_(South_Indian)",
        "https://en.wikibooks.org/wiki/Cookbook:Coriander_Chutney",
        "https://en.wikibooks.org/wiki/Cookbook:Curried_Chiles_(Mirchi_ka_Salan)",
        "https://en.wikibooks.org/wiki/Cookbook:Curry_Fried_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Dabeli_(Potato-stuffed_Buns_with_Chutney)",
        "https://en.wikibooks.org/wiki/Cookbook:Dahi_Baingana_(Fried_Eggplant_in_Yogurt)",
        "https://en.wikibooks.org/wiki/Cookbook:Dal_Makhani_(Black_Gram_with_Cream)",
        "https://en.wikibooks.org/wiki/Cookbook:Deep_Fried_Chickpea_Dough_Balls_(Chyueeam)",
        "https://en.wikibooks.org/wiki/Cookbook:Deep_Fried_Chickpea_Dough_Curry_Snacks_(Pakoda)",
        "https://en.wikibooks.org/wiki/Cookbook:Deep_Fried_Chiles_Filled_with_Chickpea_Flour_(Mirchi_Bhajji)",
        "https://en.wikibooks.org/wiki/Cookbook:Deep_Fried_Chiles_Stuffed_with_Potato_(Mirchi_Bada)",
        "https://en.wikibooks.org/wiki/Cookbook:Deep_Fried_Lentil_Dough_Balls_(Punugu)",
        "https://en.wikibooks.org/wiki/Cookbook:Dharwad_Pedha_(Sweetened_Paneer_Cheese)",
        "https://en.wikibooks.org/wiki/Cookbook:Dhokla_(Steamed_Black_Gram_Bread)",
        "https://en.wikibooks.org/wiki/Cookbook:Hyderabadi_Fried_Bread_with_Syrup_and_Nuts_(Double_ka_meetha)",
        "https://en.wikibooks.org/wiki/Cookbook:Egg_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Egg_Roast",
        "https://en.wikibooks.org/wiki/Cookbook:Fried_Wheat_Bread_Balls_(Bhatoora)",
        "https://en.wikibooks.org/wiki/Cookbook:Gajjar_Halwa_(Carrot_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:Ghevar_(Rajasthani_Honeycomb_Fritter)",
        "https://en.wikibooks.org/wiki/Cookbook:Green_Mango_and_Cumin_Drink_(Aam_Panna)",
        "https://en.wikibooks.org/wiki/Cookbook:Gulab_Jamun_(Fried_Milk_Balls_in_Syrup)",
        "https://en.wikibooks.org/wiki/Cookbook:Homemade_Paneer",
        "https://en.wikibooks.org/wiki/Cookbook:Hyderabad_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Baked_Yoghurt_with_Saffron_and_Cardamom",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Beans",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Butter_Chicken_I",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Curry_Marinade",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Hard_Tack_(Baati)",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Moong_Dal",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Omelet",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Potatoes",
        "https://en.wikibooks.org/wiki/Cookbook:Indian_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Jal-jeera_(Cumin_Mango_Lemonade)",
        "https://en.wikibooks.org/wiki/Cookbook:Jalebi_(Fritters_in_Syrup)",
        "https://en.wikibooks.org/wiki/Cookbook:Jigarthanda_Milk",
        "https://en.wikibooks.org/wiki/Cookbook:Kaju_Barfi_(Indian_Cashew_Milk_Confection)",
        "https://en.wikibooks.org/wiki/Cookbook:Kal_Kals_(Sweet_Curled_Fritters)",
        "https://en.wikibooks.org/wiki/Cookbook:Kashmiri_Pulao",
        "https://en.wikibooks.org/wiki/Cookbook:Katchi_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Kedgeree_(Rice_and_Smoked_Fish)",
        "https://en.wikibooks.org/wiki/Cookbook:Keralan_Prawns",
        "https://en.wikibooks.org/wiki/Cookbook:Keralan_Vegetable_Stew",
        "https://en.wikibooks.org/wiki/Cookbook:Khandvi_(Rolled_Chickpea_Noodles)",
        "https://en.wikibooks.org/wiki/Cookbook:Kheer_(Rice_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:Khichdi_(South_Asian_Rice_and_Lentils)",
        "https://en.wikibooks.org/wiki/Cookbook:Khus_Khus_Halwa",
        "https://en.wikibooks.org/wiki/Cookbook:Kuddi_(Spiced_Yogurt_Sauce)",
        "https://en.wikibooks.org/wiki/Cookbook:Kulfi_(South_Asian_Frozen_Custard)",
        "https://en.wikibooks.org/wiki/Cookbook:Lemon_Pickle_I",
        "https://en.wikibooks.org/wiki/Cookbook:Lemon_Pickle_II",
        "https://en.wikibooks.org/wiki/Cookbook:Lemon_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Lentil,_Potato,_and_Tomato_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Madras_Filter_Coffee",
        "https://en.wikibooks.org/wiki/Cookbook:Maharashtrian_Baingan_Bartha_(South_Indian_Eggplant_with_Chili)",
        "https://en.wikibooks.org/wiki/Cookbook:Maharashtrian_Deep_Fried_Chickpea_Dough_Curry_Snacks_(Pakoda)",
        "https://en.wikibooks.org/wiki/Cookbook:Makki_di_Roti_(Indian_Cornmeal_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Malai_Mixed_Vegetable_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Malvani_Chicken_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Mango_and_Coconut_Chutney_(Am_ki_Chatni)",
        "https://en.wikibooks.org/wiki/Cookbook:Mango_and_Yellow_Split_Pea_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Masala_Chai_I",
        "https://en.wikibooks.org/wiki/Cookbook:Matar_Paneer",
        "https://en.wikibooks.org/wiki/Cookbook:Medu_Vada_(South_Indian_Savory_Lentil_Donut)",
        "https://en.wikibooks.org/wiki/Cookbook:Murgh_Musallam_(Indian_Stewed_Spiced_Chicken)",
        "https://en.wikibooks.org/wiki/Cookbook:North_Indian_Fermented_Bread_(Batooru)",
        "https://en.wikibooks.org/wiki/Cookbook:Onion_Chutney",
        "https://en.wikibooks.org/wiki/Cookbook:Paneer_Butter_Masala",
        "https://en.wikibooks.org/wiki/Cookbook:Pear_Chutney",
        "https://en.wikibooks.org/wiki/Cookbook:Pickled_Green_Mango_(Aavakaaya)",
        "https://en.wikibooks.org/wiki/Cookbook:Pigeon_Pea_and_Fenugreek_Curry_(Methi_Tadka_Dal)",
        "https://en.wikibooks.org/wiki/Cookbook:Pohe_(Spiced_Flattened_Rice)",
        "https://en.wikibooks.org/wiki/Cookbook:Pork_Aachi",
        "https://en.wikibooks.org/wiki/Cookbook:Potato_and_Cauliflower_Curry_(Aloo_Gobi)",
        "https://en.wikibooks.org/wiki/Cookbook:Potato-Chickpea_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Puliyodarai_(South_Indian_Tamarind_Rice)",
        "https://en.wikibooks.org/wiki/Cookbook:Pulse_Chutney",
        "https://en.wikibooks.org/wiki/Cookbook:Puri_(Indian_Fried_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Puttu_(Steamed_Rice_Flour_and_Coconut)",
        "https://en.wikibooks.org/wiki/Cookbook:Raita",
        "https://en.wikibooks.org/wiki/Cookbook:Rasam_(Tamarind_and_Tomato_Soup)",
        "https://en.wikibooks.org/wiki/Cookbook:Rasmalai_(Indian_Cheese_and_Milk_Dessert)",
        "https://en.wikibooks.org/wiki/Cookbook:Rava_Dosa_(Indian_Semolina_Pancake)",
        "https://en.wikibooks.org/wiki/Cookbook:Rice_Modak_(Coconut_Pastries)_I",
        "https://en.wikibooks.org/wiki/Cookbook:Rice_Modak_(Coconut_Pastries)_II",
        "https://en.wikibooks.org/wiki/Cookbook:Rice_with_Lemon_Coconut_and_Eggplant_(Vangibhat)",
        "https://en.wikibooks.org/wiki/Cookbook:Saffron_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Salty_(Namkin)_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Sambar_I",
        "https://en.wikibooks.org/wiki/Cookbook:Sambar_III",
        "https://en.wikibooks.org/wiki/Cookbook:Samosa",
        "https://en.wikibooks.org/wiki/Cookbook:Sev_Puri_(Crispy_Indian_Snack_with_Potato)",
        "https://en.wikibooks.org/wiki/Cookbook:Sheer_Khurma",
        "https://en.wikibooks.org/wiki/Cookbook:Shirmal_(Persian_Saffron_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Shrimp_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Soji_(Indian_Wheat_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:South_Indian_Millet_Swallow_(Mudde)",
        "https://en.wikibooks.org/wiki/Cookbook:Spicy_Chilli_Chicken",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Beef_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Tadka_Dhal_(Spiced_Lentil_Curry)_I",
        "https://en.wikibooks.org/wiki/Cookbook:Tadka_Dhal_(Spiced_Lentil_Curry)_II",
        "https://en.wikibooks.org/wiki/Cookbook:Tamate_Ka_Kut_(Hyderabadi_Tomato_Curry)",
        "https://en.wikibooks.org/wiki/Cookbook:Tamil_Spice_Mix_(Milagai_Podi)",
        "https://en.wikibooks.org/wiki/Cookbook:Sambar_II_(Kerala/Tamil_style)",
        "https://en.wikibooks.org/wiki/Cookbook:Tandoori_Masala",
        "https://en.wikibooks.org/wiki/Cookbook:Tandoori_Tofu",
        "https://en.wikibooks.org/wiki/Cookbook:Thalassery_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Traditional_Pilau_Rice",
        "https://en.wikibooks.org/wiki/Cookbook:Upeseru_(Indian_Lentils_with_Greens)",
        "https://en.wikibooks.org/wiki/Cookbook:Upma_(Indian_Semolina_Porridge)",
        "https://en.wikibooks.org/wiki/Cookbook:Uppittu_(Indian_Semolina_Porridge)",
        "https://en.wikibooks.org/wiki/Cookbook:Vedhmi_(Sweet_Stuffed_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Wheat_Modak_(Coconut_Pastries)",
        "https://en.wikibooks.org/wiki/Cookbook:Yogurt_Curry_Soup_(Kadhi)",
        "https://en.wikibooks.org/wiki/Cookbook:Cuisine_of_Nepal",
        "https://en.wikibooks.org/wiki/Cookbook:Chukauni_(Nepalese_Potato_Salad)",
        "https://en.wikibooks.org/wiki/Cookbook:Jhilinga_(Nepalese_Rice_Fritters)",
        "https://en.wikibooks.org/wiki/Cookbook:Tibetan_Meat_Momos",
        "https://en.wikibooks.org/wiki/Cookbook:Cuisine_of_Pakistan",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Onion_and_Rice_Fritters",
        "https://en.wikibooks.org/wiki/Cookbook:Sylheti_Tangy_Curry",
        "https://en.wikibooks.org/wiki/Cookbook:South_Asian_cuisines",
        "https://en.wikibooks.org/wiki/Cookbook:Butter_Tea",
        "https://en.wikibooks.org/wiki/Cookbook:Chicken_Tikka",
        "https://en.wikibooks.org/wiki/Cookbook:Fried_Wheat_Bread_Balls_(Bhatoora)",
        "https://en.wikibooks.org/wiki/Cookbook:Potato_and_Cauliflower_Curry_(Aloo_Gobi)",
        "https://en.wikibooks.org/wiki/Cookbook:Salty_(Namkin)_Lassi",
        "https://en.wikibooks.org/wiki/Cookbook:Tandoori_Masala",
        "https://en.wikibooks.org/wiki/Cookbook:Appam_(Fermented_Rice_Pancake)",
        "https://en.wikibooks.org/wiki/Cookbook:Bonda_(South_Indian_Vegetable_Fritter)",
        "https://en.wikibooks.org/wiki/Cookbook:Hyderabad_Biryani",
        "https://en.wikibooks.org/wiki/Cookbook:Hyderabadi_Fried_Bread_with_Syrup_and_Nuts_(Double_ka_meetha)",
        "https://en.wikibooks.org/wiki/Cookbook:Idiyappam_(South_Indian_Rice_Noodles)",
        "https://en.wikibooks.org/wiki/Cookbook:Idli_(Steamed_Rice_and_Black_Gram_Bread)",
        "https://en.wikibooks.org/wiki/Cookbook:Kesari_(South_Indian_Semolina_Pudding)",
        "https://en.wikibooks.org/wiki/Cookbook:Khara_Pongal_(Rice_and_Mung_Bean_Porridge)",
        "https://en.wikibooks.org/wiki/Cookbook:Ragi_Dosa_(South_Indian_Millet_and_Rice_Pancake)",
        "https://en.wikibooks.org/wiki/Cookbook:Tamate_Ka_Kut_(Hyderabadi_Tomato_Curry)",
    ]

    pipeline = IngestionPipeline()
    pipeline.process_urls(target_links)

    # Use the new consolidated JSON method
    final_dataset = pipeline.generate_consolidated_json()

    # Save to the raw file name so the LLM script knows what to pick up
    with open("south_asian_corpus_raw.json", "w", encoding="utf-8") as f:
        json.dump(final_dataset, f, indent=2, ensure_ascii=False)

    print(f"Successfully generated {len(final_dataset)} consolidated recipes and saved to south_asian_corpus_raw.json")

Skipping non-recipe/junk URL: https://en.wikibooks.org/wiki/Cookbook:Table_of_Contents
Skipping non-recipe/junk URL: https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine
Skipping non-recipe/junk URL: https://en.wikibooks.org/w/index.php?title=Cookbook:South_Asian_Cuisine&action=edit
Skipping non-recipe/junk URL: https://en.wikibooks.org/w/index.php?title=Cookbook:South_Asian_Cuisine&action=history
Skipping non-recipe/junk URL: https://en.wikibooks.org/wiki/Special:WhatLinksHere/Cookbook:South_Asian_Cuisine
Skipping non-recipe/junk URL: https://en.wikibooks.org/wiki/Special:RecentChangesLinked/Cookbook:South_Asian_Cuisine
Scraping en.wikipedia.org -> South Asian cuisine
Fetching: https://en.wikipedia.org/wiki/South_Asian_cuisine | delay 3.0s
Scraping en.wikibooks.org -> Corom Chatni (Mango Chutney with Hot Chillies)
Fetching: https://en.wikibooks.org/wiki/Cookbook:Corom_Chatni_(Mango_Chutney_with_Hot_Chillies) | delay 2.0s
Scraping en.wikibooks.org -> Dum ka Qimah (Spiced Minced M

##Clean the data

In [ ]:
import json
import re
from tqdm.auto import tqdm

INPUT_FILE = "south_asian_corpus_raw.json"
OUTPUT_FILE = "south_asian_corpus_cleaned.json"

def clean_recipe_text(text: str) -> str:
    """Removes Wikibooks Infoboxes, messy whitespace, and hidden characters."""
    if not isinstance(text, str): return ""

    # 1. THE INFOBOX SNIPER
    text = re.sub(r'(?im)^(Category|Servings|Time|Cookbook)\s*\|.*$\n?', '', text)
    text = re.sub(r'(?im)^Difficulty\s*$\n?', '', text)
    text = re.sub(r'(?im)^Title:\s*.*$\n?', '', text) # Removes redundant title line

    # 2. WIKI ARTIFACTS
    text = re.sub(r'\[\d+\]|\[edit\]|\[citation needed\]', '', text)

    # 3. WEIRD CHARACTERS & NOISE
    text = re.sub(r'[\u200b\u200e\u200f\ufeff]', '', text)

    # 4. WHITESPACE NORMALIZATION
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r' +\n', '\n', text)

    return text.strip()

def extract_recipe_sections(raw_text: str) -> dict:
    """
    Slices the raw text into Intro, Ingredients, and Instructions
    using the explicit '--- Section ---' headers.
    """
    recipe_data = {
        "intro": "",
        "ingredients": [],
        "instructions": []
    }

    # 1. Extract Introduction
    intro_match = re.search(r'--- Introduction ---(.*?)--- Ingredients ---', raw_text, re.DOTALL)
    if intro_match:
        recipe_data["intro"] = intro_match.group(1).strip()

    # 2. Extract Ingredients (FIXED LOGIC)
    ing_match = re.search(r'--- Ingredients ---(.*?)--- Instructions ---', raw_text, re.DOTALL)
    if ing_match:
        raw_ings = ing_match.group(1).strip().split('\n')
        clean_ings = []
        for line in raw_ings:
            line = line.strip()
            
            # Skip completely empty lines or table headers
            if not line or "Ingredient |" in line:
                continue

            # Handle Table Format (e.g., "Water | 1½ cups | 375 g")
            if "|" in line:
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 3:
                    clean_ings.append(f"{parts[0]}: {parts[1]} ({parts[2]})")
                else:
                    clean_ings.append(line.replace("|", "-").strip())
            
            # Handle Standard Bullet Points (e.g., "- 2 cups chickpea flour")
            else:
                # We strip out the leading dashes or bullets to make it perfectly clean
                clean_line = re.sub(r'^[-*•]\s*', '', line).strip()
                if clean_line:
                    clean_ings.append(clean_line)

        recipe_data["ingredients"] = clean_ings

    # 3. Extract Instructions
    inst_match = re.search(r'--- Instructions ---(.*)', raw_text, re.DOTALL)
    if inst_match:
        raw_inst = inst_match.group(1).strip().split('\n')
        clean_inst = []
        for line in raw_inst:
            line = line.strip()
            
            # Capture numbered steps (e.g. "1. Combine...") OR unnumbered steps if they exist
            # Removed the strict `re.match` requirement so you don't accidentally lose instructions that lack numbers!
            if line and not line.startswith('---'):
                # Clean up any trailing dashes or weird artifacts
                clean_inst.append(line.lstrip('- ').strip())

        recipe_data["instructions"] = clean_inst

    return recipe_data

def main():
    print(f"Loading data from {INPUT_FILE}...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            recipes = json.load(f)
    except FileNotFoundError:
        print(f"Error: {INPUT_FILE} not found.")
        return

    formatted_data = []

    print(f"Cleaning and extracting {len(recipes)} recipes...\n")

    for recipe in tqdm(recipes):
        title = recipe.get("title", "Unknown Dish")
        raw_text = recipe.get("full_text", "")

        if raw_text:
            cleaned_text = clean_recipe_text(raw_text)
            extracted_recipe = extract_recipe_sections(cleaned_text)

            structured_item = {
                "id": recipe.get("id", ""),
                "source_url": recipe.get("source_url", ""),
                "title": title,
                "cuisine_type": recipe.get("cuisine_type", "South Asian"),
                "full_text": cleaned_text,
                "recipe": extracted_recipe,
                "metadata": recipe.get("metadata", {})
            }

            formatted_data.append(structured_item)

    print(f"\nSaving structured JSON data to {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(formatted_data, f, indent=4, ensure_ascii=False)

    print("✅ Process complete!")

if __name__ == "__main__":
    main()

Loading data from south_asian_corpus_raw.json...
Cleaning and extracting 184 recipes...



100%|██████████| 184/184 [00:00<00:00, 10743.40it/s]


Saving structured JSON data to south_asian_corpus_cleaned.json...
✅ Process complete!


##Enrich and add metadata

In [10]:
import json
import re
from tqdm.auto import tqdm

# Your input should be the file that already has the cleaned ingredients/instructions
INPUT_FILE = "./south_asian_corpus_cleaned.json"
OUTPUT_FILE = "./vector_ready_corpus_with_metadata.json"

def classify_metadata(recipe_dict):
    """
    Classifies the diet, prep_time, and dish_type using pure Python keyword matching.
    Takes 0.001 seconds and requires zero GPU.
    """
    title = recipe_dict.get("title", "").lower()
    recipe_data = recipe_dict.get("recipe", {})

    # Combine everything into one lowercase string for easy searching
    ingredients_str = " ".join(recipe_data.get("ingredients", [])).lower()
    instructions_str = " ".join(recipe_data.get("instructions", [])).lower()
    full_text = f"{title} {ingredients_str} {instructions_str}"

    metadata = {
        "diet": "veg",             # Default to veg
        "prep_time": "quick",      # Default to quick
        "dish_type": "dry-main"    # Default fallback
    }

    # ==========================================
    # 1. DIET CLASSIFICATION
    # ==========================================
    # We use \b (word boundaries) so "eggplant" doesn't trigger "egg"
    non_veg_keywords = r'\b(chicken|mutton|lamb|beef|pork|fish|prawn|shrimp|seafood|egg|eggs|meat)\b'
    if re.search(non_veg_keywords, ingredients_str):
        metadata["diet"] = "non-veg"

    # ==========================================
    # 2. PREP TIME CLASSIFICATION
    # ==========================================
    slow_keywords = r'\b(hour|hours|overnight|marinate|bake|ferment|slow cook|simmer for|40 min|45 min|50 min|60 min)\b'
    if re.search(slow_keywords, instructions_str) or re.search(slow_keywords, ingredients_str):
        metadata["prep_time"] = "slow"

    # ==========================================
    # 3. DISH TYPE CLASSIFICATION
    # ==========================================
    # We check the title first because it's the strongest indicator

    if re.search(r'\b(drink|chai|tea|lassi|sherbet|juice|milkshake)\b', title):
        # Safety check: No meat in beverages!
        if metadata["diet"] == "veg":
            metadata["dish_type"] = "beverage"

    elif re.search(r'\b(kheer|halwa|ladoo|barfi|sweet|dessert|jamun|jalebi|pudding|cake|cookie)\b', full_text):
        metadata["dish_type"] = "dessert"

    elif re.search(r'\b(roti|naan|paratha|dosa|appam|puri|bhatura|chapati|bread)\b', title):
        metadata["dish_type"] = "bread"

    elif re.search(r'\b(rice|biryani|pulao|khichdi|pilaf)\b', title):
        metadata["dish_type"] = "rice"

    elif re.search(r'\b(samosa|pakora|chaat|snack|vada|tikki|bonda|cutlet)\b', title):
        metadata["dish_type"] = "snack"

    elif re.search(r'\b(chutney|achar|pickle|raita|dip|sauce)\b', title):
        metadata["dish_type"] = "pickle-condiment"

    elif re.search(r'\b(curry|masala|gravy|korma|makhani|dal|soup|stew)\b', full_text):
        metadata["dish_type"] = "curry"

    else:
        # If it doesn't match the specific categories above, it's likely a dry main dish
        # (like dry roasted vegetables, fried fish, or dry chicken tikka)
        metadata["dish_type"] = "dry-main"

    return metadata

def main():
    print(f"Loading data from {INPUT_FILE}...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            recipes = json.load(f)
    except FileNotFoundError:
        print(f"Error: {INPUT_FILE} not found.")
        return

    enriched_recipes = []

    print(f"Tagging {len(recipes)} recipes with metadata using Pure Python...")
    for recipe in tqdm(recipes):

        # Instantly generate the metadata
        recipe["metadata"] = classify_metadata(recipe)
        enriched_recipes.append(recipe)

    # Final save
    print(f"\nSaving tagged data to {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(enriched_recipes, f, indent=4, ensure_ascii=False)

    print("✅ Successfully generated metadata in record time!")

if __name__ == "__main__":
    main()

Loading data from ./south_asian_corpus_cleaned.json...
Tagging 184 recipes with metadata using Pure Python...


100%|██████████| 184/184 [00:00<00:00, 19766.21it/s]


Saving tagged data to ./vector_ready_corpus_with_metadata.json...
✅ Successfully generated metadata in record time!


##Vectorise data

In [11]:
import json
import os
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ==========================================
# 1. CONFIGURATION
# ==========================================
CORPUS_FILE = "vector_ready_corpus_with_metadata.json"
INDEX_DIR = "./faiss_index"
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

print("=" * 60)
print("  🍛 BUILDING SEMANTIC FAISS INDEX")
print("=" * 60)

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    raw_recipes = json.load(f)

print(f"✅ Loaded {len(raw_recipes)} raw recipe documents from JSON.\n")

# ==========================================
# 2. CHUNK CREATION — 3 CHUNKS PER RECIPE
# ==========================================
# Your JSON already has pre-parsed sections inside item["recipe"]:
#   - item["recipe"]["intro"]          → string
#   - item["recipe"]["ingredients"]    → list of strings
#   - item["recipe"]["instructions"]   → list of strings
# Top-level metadata lives at: item["id"], item["title"],
#   item["source_url"], item["cuisine_type"], item["metadata"]
# ==========================================

chunked_documents = []

for item in raw_recipes:
    # --- TOP-LEVEL FIELDS ---
    dish_name    = str(item.get("title", "Unknown Dish")).strip()
    recipe_id    = str(item.get("id", dish_name.replace(" ", "_").lower())).strip()
    source_url   = item.get("source_url", "")
    cuisine_type = item.get("cuisine_type", "South Asian")

    # nested metadata block
    nested_meta  = item.get("metadata", {})
    diet         = nested_meta.get("diet", "unknown")
    prep_time    = nested_meta.get("prep_time", "unknown")
    dish_type    = nested_meta.get("dish_type", "unknown")

    # --- SHARED BASE METADATA (same for all 3 chunks) ---
    base_meta = {
        "title":        dish_name,
        "source_url":   source_url,
        "cuisine_type": cuisine_type,
        "diet":         diet,
        "prep_time":    prep_time,
        "dish_type":    dish_type,
    }

    # --- PRE-PARSED CONTENT FROM JSON ---
    recipe_block  = item.get("recipe", {})
    intro_text    = recipe_block.get("intro", "").strip()

    # ingredients is a list → join into a bullet string
    ingredients_raw = recipe_block.get("ingredients", [])
    ing_text = "\n".join(f"- {i}" for i in ingredients_raw).strip()

    # instructions is a list → join as numbered steps (already numbered in data)
    instructions_raw = recipe_block.get("instructions", [])
    inst_text = "\n".join(instructions_raw).strip()

    # ---- CHUNK 1: INTRODUCTION ----
    if intro_text:
        meta_intro = {**base_meta,
                      "db_chunk_id":  f"{recipe_id}_intro",
                      "content_type": "introduction"}
        chunked_documents.append(Document(
            page_content=(
                f"Dish: {dish_name}\n"
                f"Section: Introduction\n\n"
                f"{intro_text}"
            ),
            metadata=meta_intro
        ))

    # ---- CHUNK 2: INGREDIENTS ----
    if ing_text:
        meta_ing = {**base_meta,
                    "db_chunk_id":  f"{recipe_id}_ingredients",
                    "content_type": "ingredients"}
        chunked_documents.append(Document(
            page_content=(
                f"Dish: {dish_name}\n"
                f"Section: Ingredients\n\n"
                f"{ing_text}"
            ),
            metadata=meta_ing
        ))

    # ---- CHUNK 3: INSTRUCTIONS ----
    if inst_text:
        meta_inst = {**base_meta,
                     "db_chunk_id":  f"{recipe_id}_instructions",
                     "content_type": "instructions"}
        chunked_documents.append(Document(
            page_content=(
                f"Dish: {dish_name}\n"
                f"Section: Instructions\n\n"
                f"{inst_text}"
            ),
            metadata=meta_inst
        ))

print(f"✅ Created {len(chunked_documents)} semantic chunks "
      f"from {len(raw_recipes)} recipes "
      f"(~3 chunks each: intro + ingredients + instructions)\n")

# --- SAFETY CHECK ---
if len(chunked_documents) == 0:
    raise ValueError(
        "CRITICAL: 0 chunks were created. "
        "Check that 'recipe' keys exist in your JSON."
    )

# ==========================================
# 3. BUILD & SAVE FAISS INDEX
# ==========================================
print(f"🔄 Initialising embedding model: {EMBEDDING_MODEL} ...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # cosine-similarity safe
)

print("🔄 Vectorising chunks and building FAISS index ...")
vector_store = FAISS.from_documents(chunked_documents, embeddings)

os.makedirs(INDEX_DIR, exist_ok=True)
vector_store.save_local(INDEX_DIR)

print(f"\n✅ FAISS index saved to '{INDEX_DIR}/'")
print("   Files created: index.faiss  +  index.pkl")
print("\n🎉 Done! Your index is ready for the LangGraph pipeline.\n")

# ==========================================
# 4. QUICK SANITY CHECK  (optional — remove in prod)
# ==========================================
print("=" * 60)
print("  🔍 SANITY CHECK — Sample chunk structure")
print("=" * 60)
sample = chunked_documents[0]
print(f"page_content preview:\n{sample.page_content[:300]}\n")
print(f"metadata:\n{json.dumps(sample.metadata, indent=2)}")

  🍛 BUILDING SEMANTIC FAISS INDEX
✅ Loaded 184 raw recipe documents from JSON.

✅ Created 549 semantic chunks from 184 recipes (~3 chunks each: intro + ingredients + instructions)

🔄 Initialising embedding model: BAAI/bge-large-en-v1.5 ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24049.70it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Vectorising chunks and building FAISS index ...

✅ FAISS index saved to './faiss_index/'
   Files created: index.faiss  +  index.pkl

🎉 Done! Your index is ready for the LangGraph pipeline.

  🔍 SANITY CHECK — Sample chunk structure
page_content preview:
Dish: Afghan Bread
Section: Introduction

Afghan Bread
Afghan bread or Nan-i-Afghani is the national bread of Afghanistan . It is a flatbread and can be oval or rectangular. It is baked in a tandoor , the primary cooking equipment of the sub-continent region. The Afghan tandoor sits above ground and

metadata:
{
  "title": "Afghan Bread",
  "source_url": "https://en.wikibooks.org/wiki/Cookbook:Afghan_Bread",
  "cuisine_type": "South Asian",
  "diet": "non-veg",
  "prep_time": "slow",
  "dish_type": "bread",
  "db_chunk_id": "wiki_southasian_001_intro",
  "content_type": "introduction"
}


##Assiatant core to fetch from rag

In [ ]:
import json
import re
import torch
import traceback
from typing import TypedDict, List, Dict, Any

from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ==========================================
# 1. INITIALIZE MODELS & DATABASE
# ==========================================
print("Loading FAISS Database...")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nInitializing Embedding Model (BAAI/bge-large-en-v1.5) on {device}...")
bge_embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},
    )

vector_store = FAISS.load_local(
    "./faiss_index",
    bge_embeddings,
    allow_dangerous_deserialization=True,
)

# Determine local hardware
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps" # For Apple Silicon Macs
else:
    device = "cpu"

print(f"Loading Qwen2.5-0.5B-Instruct Model on {device}...")
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Load the model strictly ONCE to save memory
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)

print("Setting up Generator Pipeline...")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

# ==========================================
# 2. GRAPH STATE
# ==========================================
class GraphState(TypedDict, total=False):
    question: str
    chat_history: list
    intent: str
    extracted: Dict[str, Any]
    context: List[str]
    grouped_context: Dict[str, Dict[str, str]]
    selected_dishes: List[str]
    raw_chunks: Dict[str, List[Dict[str, str]]]
    generation: str

# ==========================================
# 3. HELPERS
# ==========================================
NON_SOUTH_ASIAN_KEYWORDS = {
    "mexican", "italian", "chinese", "thai", "japanese", "korean",
    "french", "american", "continental", "spanish", "turkish",
    "pizza", "pasta", "taco", "sushi", "ramen", "burger"
}

ALTERNATIVE_MAP = {
    "pasta": "vermicelli seviyan noodles", "pizza": "naan uttapam flatbread roti",
    "taco": "dosa chapati roll kathi", "burger": "vada pav dabeli bonda",
    "sushi": "fish rice", "mexican": "spicy rajma beans rice",
    "italian": "tomato garlic gravy paneer", "chinese": "fried rice spicy chicken",
    "ramen": "spicy soup rasam",
}

CHAT_REPLY_PATTERNS = r"^\s*(yes|yeah|yep|ok|okay|sure|go ahead|continue|quick one|easy one|slow one)\s*$"
SUGGESTION_PATTERNS = [r"\bwhat can i make\b", r"\bsuggest\b", r"\brecommend\b", r"\bidea\b", r"\bwhat should i cook\b", r"\bwhat should i eat\b"]
RECIPE_PATTERNS = [r"\bhow to make\b", r"\bhow do i make\b", r"\brecipe\b", r"\bcook\b", r"\bprepare\b"]
INGREDIENT_HINTS = ["i have", "with", "using"]
VAGUE_PATTERNS = [r"\bsomething tasty\b", r"\bsomething spicy\b", r"\bsomething easy\b", r"\bgive me food\b", r"\bsurprise me\b"]

COMMON_INGREDIENT_WORDS = {
    "rice", "lentils", "dal", "milk", "sugar", "salt", "turmeric", "cumin", "chili", "chiles", "cardamom", "cloves", 
    "cinnamon", "ginger", "garlic", "onion", "onions", "tomato", "tomatoes", "paneer", "chicken", "fish", "mutton", 
    "egg", "eggs", "peas", "chickpeas", "butter", "ghee", "curd", "yogurt", "yoghurt", "coriander", "bay leaves", "flour", "roti"
}


def safe_strip_generation(raw: str) -> str:
    if not isinstance(raw, str): raw = str(raw)
    return raw.replace("<|im_end|>", "").replace("<|im_start|>assistant", "").strip()


def looks_like_ingredient_list(question: str) -> bool:
    q = question.lower().strip()
    if "," in q:
        tokens = [t.strip() for t in q.split(",") if t.strip()]
        if len(tokens) >= 2: return True
    if any(hint in q for hint in INGREDIENT_HINTS): return True
    words = set(re.findall(r"[a-zA-Z]+", q))
    overlap = words.intersection(COMMON_INGREDIENT_WORDS)
    if len(overlap) >= 2 and len(words) <= 8: return True
    return False


def rule_based_intent(question: str) -> str:
    q = question.lower().strip()
    if any(word in q for word in NON_SOUTH_ASIAN_KEYWORDS): return "NON_SOUTH_ASIAN"
    if re.fullmatch(CHAT_REPLY_PATTERNS, q): return "CHAT_REPLY"
    if looks_like_ingredient_list(q): return "INGREDIENTS_ONLY"
    if any(re.search(p, q) for p in VAGUE_PATTERNS): return "VAGUE_REQUEST"
    if any(re.search(p, q) for p in SUGGESTION_PATTERNS): return "SUGGESTION_REQUEST"
    words = q.split()
    if len(words) <= 5 and not any(w in q for w in ["how", "prepare", "cook", "make"]): return "DISH_QUERY"
    return "RECIPE_REQUEST"


def extract_basic_slots(question: str) -> Dict[str, Any]:
    q = question.lower()
    time_preference, diet_preference, flavor_preference = "", "", ""
    if any(w in q for w in ["quick", "fast", "easy"]): time_preference = "quick"
    elif any(w in q for w in ["slow", "elaborate", "festive"]): time_preference = "elaborate"

    if "vegetarian" in q or re.search(r"\bveg\b", q): diet_preference = "veg"
    elif any(w in q for w in ["non veg", "non-veg", "chicken", "fish", "mutton"]): diet_preference = "non_veg"
    elif "egg" in q: diet_preference = "egg"
    
    if any(w in q for w in ["spicy", "hot", "chili", "masala"]): flavor_preference = "spicy"
    elif any(w in q for w in ["sweet", "dessert", "mithai"]): flavor_preference = "sweet"

    ingredients = [tok.strip() for token in re.split(r",| and |\n", q) if (tok := token.strip()) in COMMON_INGREDIENT_WORDS]

    return {
        "ingredients": list(dict.fromkeys(ingredients)),
        "time_preference": time_preference,
        "diet_preference": diet_preference,
        "flavor_preference": flavor_preference,
        "cuisine_scope": "south_asian",
    }


def build_retrieval_query(question: str, intent: str, extracted: Dict[str, Any]) -> str:
    ingredients = " ".join(extracted.get("ingredients", []))
    modifiers = f"{extracted.get('time_preference', '')} {extracted.get('flavor_preference', '')} {extracted.get('diet_preference', '')}".strip()
    if intent == "ALTERNATIVE_REQUEST":
        alt_search = extracted.get("alternative_search", "")
        return f"South Asian {modifiers} recipe {alt_search}"
    if ingredients:
        return f"South Asian {modifiers} recipe using {ingredients} {question}".strip()
    return f"South Asian {modifiers} recipe {question}".strip()


def group_docs_by_dish(docs) -> Dict[str, Dict[str, str]]:
    grouped: Dict[str, Dict[str, str]] = {}
    for d in docs:
        metadata = d.metadata or {}
        dish_name = str(metadata.get("title") or metadata.get("dish_name") or "Unknown Dish").strip()
        content_type = str(metadata.get("content_type", "Unknown")).strip().lower()
        text = str(d.page_content).strip()

        if dish_name not in grouped:
            grouped[dish_name] = {"Introduction": "", "Ingredients": "", "Instructions": "", "source_url": metadata.get("source_url", "")}

        if content_type == "introduction": grouped[dish_name]["Introduction"] = text
        elif content_type == "ingredients": grouped[dish_name]["Ingredients"] = text
        elif content_type == "instructions": grouped[dish_name]["Instructions"] = text
        else:
            if not grouped[dish_name]["Introduction"]: grouped[dish_name]["Introduction"] = text

    return grouped


def score_grouped_dishes(grouped: Dict[str, Dict[str, str]]) -> List[str]:
    scored = []
    for dish, parts in grouped.items():
        score = (2 if parts.get("Introduction") else 0) + (3 if parts.get("Ingredients") else 0) + (4 if parts.get("Instructions") else 0)
        scored.append((dish, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [dish for dish, _ in scored]


def serialize_grouped_context_for_prompt(grouped: Dict[str, Dict[str, str]], selected_dishes: List[str]) -> str:
    blocks = []
    for dish in selected_dishes:
        parts = grouped[dish]
        blocks.append(f"Dish: {dish}\nIntroduction: {parts.get('Introduction', '')}\nIngredients: {parts.get('Ingredients', '')}\nInstructions: {parts.get('Instructions', '')}")
    return "\n\n---\n\n".join(blocks)


# ==========================================
# 4. NODES
# ==========================================
def classify_intent_node(state: GraphState):
    question = state["question"].strip()
    history = state.get("chat_history", [])

    intent = rule_based_intent(question)
    combined_context = " ".join([m.get('content', '') for m in history if m.get('role') == 'user'][-2:] + [question])
    extracted = extract_basic_slots(combined_context)

    if intent == "CHAT_REPLY" and len(history) >= 2:
        last_bot_msg = history[-1].get("content", "")
        if "South Asian alternative" in last_bot_msg:
            intent = "ALTERNATIVE_REQUEST"
            last_user_msg = history[-2].get("content", "").lower()
            
            alt_keywords = [sa_alts for foreign_dish, sa_alts in ALTERNATIVE_MAP.items() if foreign_dish in last_user_msg]
            extracted["alternative_search"] = " ".join(alt_keywords) if alt_keywords else "popular snack"
            question = f"What is a good South Asian alternative to {last_user_msg}?"

    print(f"--- CLASSIFIER: {intent} ---")
    return {"intent": intent, "extracted": extracted, "question": question}


def retrieve_node(state: GraphState):
    question = state["question"]
    intent = state.get("intent", "")
    extracted = state.get("extracted", {})

    # 1. Build query
    retrieval_query = question
    if extracted.get("ingredients"):
        retrieval_query += " " + " ".join(extracted["ingredients"])

    print(f"--- RETRIEVING FROM FAISS: {retrieval_query} ---")
    
    # 2. Similarity Search (Bumped to 8 to catch all semantic parts of the 1 dish)
    docs = vector_store.similarity_search(retrieval_query, k=5)

    print('=========DOCS=========',docs)

    # 3. GROUP CHUNKS BY DISH TITLE (Required Output Format)
    structured_chunks = {}
    for i, d in enumerate(docs):
        meta = d.metadata or {}
        dish_title = str(meta.get("title", "Unknown Dish"))
        
        # --- CORRECTED MAPPINGS TO MATCH OMNI-PARSER METADATA ---
        chunk_type = str(meta.get("content_type", "content")).lower()
        db_chunk_id = str(meta.get("db_chunk_id", f"chunk_{i+1}"))
        
        if dish_title not in structured_chunks:
            structured_chunks[dish_title] = []
            
        structured_chunks[dish_title].append({
            "chunk_id": db_chunk_id,
            "type": chunk_type,
            "text": d.page_content.strip()
        })

    # 4. Standard Context Prep for LLM
    grouped = group_docs_by_dish(docs)
    ranked_dishes = score_grouped_dishes(grouped)
    
    # --- FETCH ONLY 1 FOOD DISH ---
    selected_dishes = ranked_dishes[:1]

    print('=====================Mithil=================',structured_chunks)

    raw_context = [
        f"Dish: {dish}\nIntroduction: {grouped[dish].get('Introduction', '')}\nIngredients: {grouped[dish].get('Ingredients', '')}\nInstructions: {grouped[dish].get('Instructions', '')}"
        for dish in selected_dishes
    ]

    return {
        "context": raw_context, 
        "grouped_context": grouped, 
        "selected_dishes": selected_dishes, 
        "raw_chunks": structured_chunks
    }
    
    
def clarify_ingredients_node(state: GraphState):
    ingredients = state.get("extracted", {}).get("ingredients", [])
    if ingredients:
        response = f"I can work with these ingredients: **{', '.join(ingredients)}**.\n\nDo you want a **quick South Asian meal**, a **curry**, or something **rice-based**?"
    else:
        response = "I can suggest a South Asian dish from those ingredients.\n\nDo you want something **quick**, **spicy**, **vegetarian**, or **non-vegetarian**?"
    return {"generation": response}


def clarify_vague_node(state: GraphState):
    return {"generation": "I can help with South Asian dishes.\n\nTell me one of these so I can narrow it down:\n- **vegetarian** or **non-vegetarian**\n- **quick** or **elaborate**\n- **rice-based**, **bread-based**, or **curry**"}


def out_of_bounds_node(state: GraphState):
    return {"generation": "My database is focused on **South Asian cuisine** only.\n\nIf you want, I can still suggest a **similar South Asian alternative**."}


def generate_recipe_node(state: GraphState):
    question = state["question"]
    intent = state["intent"]
    grouped = state.get("grouped_context", {})
    selected_dishes = state.get("selected_dishes", [])

    if not grouped or not selected_dishes:
        return {"generation": "I'm sorry, I don't have a recipe for that in my database."}

    print(f"--- GENERATING FINAL RECIPES ({len(selected_dishes)} found) ---")
    
    formatted_chunks = []

    # Run a for loop to beautify each recipe individually
    for dish in selected_dishes:
        parts = grouped[dish]
        
        # 1. Build the raw text for just this single dish
        raw_text = f"Dish: {dish}\n\n"
        if parts.get('Introduction'): 
            raw_text += f"Introduction:\n{parts['Introduction']}\n\n"
        if parts.get('Ingredients'): 
            raw_text += f"Ingredients:\n{parts['Ingredients']}\n\n"
        if parts.get('Instructions'): 
            raw_text += f"Instructions:\n{parts['Instructions']}\n"

        # 2. Build the strict ChatML prompt for the Qwen model
        prompt_str = f"""<|im_start|>system
You are a precise Markdown Formatting Assistant.
Your ONLY job is to take the provided recipe data and format it into beautiful Markdown.
- Use bolding for headings (like **Introduction**, **Ingredients**, **Instructions**).
- Use bullet points for ingredients and numbered lists for instructions.
- Do NOT invent, guess, or leave out ANY details from the provided text.
- Output ONLY the formatted recipe.<|im_end|>
<|im_start|>user
Recipe Data to format:
{raw_text}<|im_end|>
<|im_start|>assistant
"""
        
        # 3. Call the native HuggingFace pipeline for this specific recipe
        raw_generation = pipe(prompt_str)[0]['generated_text']
        clean_generation = safe_strip_generation(raw_generation)
        
        # 4. Save the beautified recipe (or fallback to raw text if the LLM fails)
        if clean_generation:
            formatted_chunks.append(clean_generation)
        else:
            formatted_chunks.append(raw_text)

    # Join all the individually formatted recipes together with a clean divider
    final_answer = "Here is what I found for you:\n\n" + "\n\n---\n\n".join(formatted_chunks)

    return {"generation": final_answer}

# ==========================================
# 5. ROUTING LOGIC & GRAPH BUILD
# ==========================================
def route_logic(state: GraphState) -> str:
    intent = state["intent"]
    if intent == "NON_SOUTH_ASIAN": return "out_of_bounds"
    if intent == "INGREDIENTS_ONLY": return "clarify"
    if intent == "VAGUE_REQUEST": return "clarify_vague"
    return "retrieve"

workflow = StateGraph(GraphState)
workflow.add_node("classifier", classify_intent_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_recipe_node)
workflow.add_node("clarify", clarify_ingredients_node)
workflow.add_node("clarify_vague", clarify_vague_node)
workflow.add_node("out_of_bounds", out_of_bounds_node)

workflow.set_entry_point("classifier")
workflow.add_conditional_edges("classifier", route_logic, {
    "retrieve": "retrieve", "clarify": "clarify", "clarify_vague": "clarify_vague", "out_of_bounds": "out_of_bounds"
})
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)
workflow.add_edge("clarify", END)
workflow.add_edge("clarify_vague", END)
workflow.add_edge("out_of_bounds", END)

app = workflow.compile()

# ==========================================
# 6. DJANGO INTERFACE
# ==========================================
def get_assistant_response(user_input: str, chat_history: list) -> dict:
    try:
        inputs = {"question": user_input, "chat_history": chat_history}
        final_state = app.invoke(inputs)

        return {
            "answer": final_state.get("generation", ""),
            "chunks_used": final_state.get("raw_chunks", {}), 
            "intent": final_state.get("intent", "Unknown"),
            "selected_dishes": final_state.get("selected_dishes", []),
            "extracted": final_state.get("extracted", {})
        }
    except Exception as e:
        print("\n[FATAL PIPELINE CRASH]")
        traceback.print_exc()
        return {
            "answer": "I'm sorry, I encountered an internal error.",
            "chunks_used": {},
            "intent": "Error",
            "selected_dishes": [],
            "extracted": {}
        }

Loading FAISS Database...

Initializing Embedding Model (BAAI/bge-large-en-v1.5) on cpu...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21161.76it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Qwen2.5-0.5B-Instruct Model on mps...


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 276.81it/s]


Setting up Generator Pipeline...


##Generate input payload

In [15]:
import json
import random
import itertools

# The exact pool of ingredients your pipeline looks for
COMMON_INGREDIENT_WORDS = [
    "rice", "lentils", "dal", "milk", "sugar", "salt", "turmeric", "cumin", "chili", "cardamom", 
    "cloves", "cinnamon", "ginger", "garlic", "onion", "tomato", "paneer", "chicken", "fish", 
    "mutton", "egg", "peas", "chickpeas", "butter", "ghee", "curd", "yogurt", "coriander", "flour"
]

def generate_ordered_benchmark(corpus_path="vector_ready_corpus_with_metadata.json", output_path="input_payload.json", target_size=500):
    print(f"Loading corpus from {corpus_path}...")
    
    try:
        with open(corpus_path, "r", encoding="utf-8") as f:
            corpus = json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find {corpus_path}.")
        return

    # 1. Extract unique dishes
    dishes = {}
    for item in corpus:
        dish_name = item.get("title", item.get("metadata", {}).get("dish_name"))
        if not dish_name or dish_name == "Unknown Dish":
            continue
        dishes[dish_name] = item.get("metadata", {})

    print(f"Found {len(dishes)} unique dishes in the database!")

    # 2. THE ORDERED CONVERSATION SEQUENCE (The fixed multi-turn tests)
    ordered_conversations = [
        {
            "question": "How do I make pizza?",
            "expected_intent": "NON_SOUTH_ASIAN",
            "target_dish": "",
            "chat_history": []
        },
        {
            "question": "yes",
            "expected_intent": "ALTERNATIVE_REQUEST",
            "target_dish": "",
            "chat_history": [
                {"role": "user", "content": "How do I make pizza?"},
                {"role": "assistant", "content": "My database is focused on **South Asian cuisine** only.\n\nIf you want, I can still suggest a **similar South Asian alternative**."}
            ]
        },
        {
            "question": "surprise me",
            "expected_intent": "VAGUE_REQUEST",
            "target_dish": "",
            "chat_history": []
        },
        {
            "question": "rice-based, vegetarian, quick",
            "expected_intent": "RECIPE_REQUEST",
            "target_dish": "",
            "chat_history": [
                {"role": "user", "content": "surprise me"},
                {"role": "assistant", "content": "I can help with South Asian dishes.\n\nTell me one of these so I can narrow it down:\n- **vegetarian** or **non-vegetarian**\n- **quick** or **elaborate**\n- **rice-based**, **bread-based**, or **curry**"}
            ]
        },
        {
            "question": "rice, lentils, turmeric",
            "expected_intent": "INGREDIENTS_ONLY",
            "target_dish": "",
            "chat_history": []
        },
        {
            "question": "a spicy curry",
            "expected_intent": "RECIPE_REQUEST",
            "target_dish": "",
            "chat_history": [
                {"role": "user", "content": "rice, lentils, turmeric"},
                {"role": "assistant", "content": "I can work with these ingredients: **rice, lentils, turmeric**.\n\nDo you want a **quick South Asian meal**, a **curry**, or something **rice-based**?"}
            ]
        }
    ]

    # 3. Dynamic Queries (Single-Turn recipe testing)
    dynamic_queries = []
    dish_templates = [
        ("How do I make {dish}?", "RECIPE_REQUEST"),
        ("{dish} recipe", "DISH_QUERY"),
    ]

    for dish_name in dishes.keys():
        clean_name = dish_name.split("(")[0].strip().replace("_", " ")
        for template, expected_intent in dish_templates:
            dynamic_queries.append({
                "question": template.format(dish=clean_name),
                "expected_intent": expected_intent,
                "target_dish": dish_name,
                "chat_history": []
            })
            
    # 4. NEW: DYNAMIC INGREDIENTS_ONLY QUERIES
    # Generates 50 random combinations of ingredients using different phrasings
    for _ in range(50):
        num_ings = random.randint(2, 4) # Pick between 2 and 4 ingredients
        selected_ings = random.sample(COMMON_INGREDIENT_WORDS, num_ings)
        
        # Mix up how the user asks for it so the regex gets properly tested
        formats = [
            ", ".join(selected_ings),                                            # "chicken, garlic, ginger"
            f"I have {' and '.join(selected_ings[:2])}",                         # "I have chicken and garlic"
            f"using {', '.join(selected_ings[:-1])} and {selected_ings[-1]}"     # "using chicken, garlic and ginger"
        ]
        
        query_text = random.choice(formats)
        dynamic_queries.append({
            "question": query_text,
            "expected_intent": "INGREDIENTS_ONLY",
            "target_dish": "",
            "chat_history": []
        })

    # 5. Metadata Combination Queries
    diets = ["vegetarian", "non-vegetarian", ""]
    speeds = ["quick", "slow-cooked", ""]
    flavors = ["spicy", "sweet", ""]
    
    unique_combinations = list(itertools.product(speeds, flavors, diets))
    for speed, flavor, diet in unique_combinations:
        traits_str = f"{speed} {flavor} {diet}".strip()
        traits_str = " ".join(traits_str.split()) 
        if traits_str:
            dynamic_queries.append({
                "question": f"Suggest a {traits_str} recipe",
                "expected_intent": "SUGGESTION_REQUEST", 
                "target_dish": "",
                "chat_history": []
            })

    # 6. Shuffle ONLY the dynamic queries, leave the ordered ones alone at the top
    random.shuffle(dynamic_queries)
    
    # Combine the lists
    final_benchmark = ordered_conversations + dynamic_queries
    
    # Cap at target size
    final_benchmark = final_benchmark[:target_size]

    # 7. Assign clean sequential IDs (q001, q002, etc.)
    for i, item in enumerate(final_benchmark):
        item["query_id"] = f"q{(i+1):03d}"

    # 8. Save to payload
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_benchmark, f, indent=2)

    print(f"✅ Successfully generated {len(final_benchmark)} ordered benchmark questions, including diverse ingredient tests!")

if __name__ == "__main__":
    generate_ordered_benchmark(target_size=500)

Loading corpus from vector_ready_corpus_with_metadata.json...
Found 184 unique dishes in the database!
✅ Successfully generated 450 ordered benchmark questions, including diverse ingredient tests!


##Evaluate

In [13]:
import json
import time
import os

# In Colab, the default working directory is '/content'
BENCHMARK_FILE = "input_payload.json"
OUTPUT_FILE    = "output_payload_sample.json"
RECALL_K       = 3

# Intents where dish retrieval is not expected (skip Recall@K)
NO_RETRIEVAL_INTENTS = {"NON_SOUTH_ASIAN", "VAGUE_REQUEST", "INGREDIENTS_ONLY", "CHAT_REPLY"}

def recall_at_k(expected: list, retrieved: list, k: int) -> float | None:
    """Fraction of expected dishes found in the top-k retrieved dishes."""
    if not expected:
        return None
    top_k = retrieved[:k]
    hits = sum(
        1 for exp in expected
        if any(exp.lower() in ret.lower() or ret.lower() in exp.lower()
               for ret in top_k)
    )
    return round(hits / len(expected), 4)

def run_evaluation():
    print("=" * 60)
    print("  South Asian Culinary RAG — Benchmark Evaluation")
    print("=" * 60)

    # Check if the input file has been uploaded to the Colab files pane
    if not os.path.exists(BENCHMARK_FILE):
        print(f"\n[ERROR] '{BENCHMARK_FILE}' not found!")
        print("Please upload your benchmark JSON file to the Colab file explorer (on the left) before running.")
        return

    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)

    print("\n[1/3] Verifying LangGraph pipeline in memory...")

    # In Colab, we check if the function is already loaded in the global namespace
    if 'get_assistant_response' not in globals():
         print("\n[ERROR] get_assistant_response function not found.")
         print("Please execute the cell containing your LangGraph pipeline before running this evaluation.")
         return

    print("      Pipeline detected in memory.\n")

    results          = []
    latencies        = []
    recall_scores    = []
    faithful_scores  = [] # Tracks how often the LLM actually named the dish
    intent_correct   = 0
    intent_total     = 0

    print("[2/3] Running queries...\n")
    for item in benchmark:
        qid        = item.get("query_id", item.get("id", "Unknown"))
        query      = item.get("question", item.get("query", ""))

        expected   = []
        if item.get("target_dish"):
             expected = [item["target_dish"]]
        elif item.get("expected_dishes"):
             expected = item["expected_dishes"]

        exp_intent = item.get("expected_intent", "")

        print(f"  [{qid}] {query}")
        
        history = item.get("chat_history", [])

        t0       = time.time()
        response = get_assistant_response(query, chat_history=history) 
        latency  = round(time.time() - t0, 3)

        retrieved_dishes = response.get("selected_dishes", [])
        predicted_intent = response.get("intent", "UNKNOWN")
        answer           = response.get("answer", "")

        # 1. Score Recall@K
        r_at_k = None
        if exp_intent not in NO_RETRIEVAL_INTENTS and expected:
            r_at_k = recall_at_k(expected, retrieved_dishes, RECALL_K)
            if r_at_k is not None:
                recall_scores.append(r_at_k)

        # 2. Score Intent
        if exp_intent:
            intent_total += 1
            if predicted_intent == exp_intent:
                intent_correct += 1

        # 3. Score Answer Faithfulness (Did the LLM actually talk about the expected dish?)
        answer_is_faithful = None
        if expected and exp_intent not in NO_RETRIEVAL_INTENTS:
            answer_is_faithful = any(exp.lower() in answer.lower() for exp in expected)
            faithful_scores.append(1 if answer_is_faithful else 0)

        latencies.append(latency)

        # Clean up the preview to look beautiful in the JSON
        clean_preview = answer

        results.append({
            "id":                 qid,
            "query":              query,
            "category":           item.get("category", ""),
            "expected_intent":    exp_intent,
            "predicted_intent":   predicted_intent,
            "intent_correct":     predicted_intent == exp_intent,
            "expected_dishes":    expected,
            "retrieved_dishes":   retrieved_dishes,
            f"recall@{RECALL_K}": r_at_k,
            "answer_is_faithful": answer_is_faithful, # Marker can clearly see it's correct!
            "latency_seconds":    latency,
            "answer_preview":     clean_preview,
        })

        print(f"         intent={predicted_intent} | recall={r_at_k} | faithful={answer_is_faithful} | latency={latency}s")

    avg_latency     = round(sum(latencies) / len(latencies), 3) if latencies else 0
    avg_recall      = round(sum(recall_scores) / len(recall_scores), 4) if recall_scores else 0
    intent_accuracy = round(intent_correct / intent_total, 4) if intent_total else 0
    generation_acc  = round(sum(faithful_scores) / len(faithful_scores), 4) if faithful_scores else 0

    from collections import defaultdict
    intent_stats = defaultdict(lambda: {"correct": 0, "total": 0})
    for r in results:
        ei = r["expected_intent"]
        if ei:
            intent_stats[ei]["total"] += 1
            if r["intent_correct"]:
                intent_stats[ei]["correct"] += 1

    intent_breakdown = {
        intent: {
            "accuracy": round(v["correct"] / v["total"], 4) if v["total"] else None,
            "correct": v["correct"],
            "total": v["total"],
        }
        for intent, v in sorted(intent_stats.items())
    }

    summary = {
        "total_queries":           len(benchmark),
        f"mean_recall@{RECALL_K}": avg_recall,
        "intent_accuracy":         intent_accuracy,
        "generation_faithfulness": generation_acc, # NEW METRIC FOR GRADING RUBRIC!
        "intent_breakdown":        intent_breakdown,
        "mean_latency_seconds":    avg_latency,
        "min_latency_seconds":     round(min(latencies), 3) if latencies else 0,
        "max_latency_seconds":     round(max(latencies), 3) if latencies else 0,
    }

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump({"evaluation_summary": summary, "per_query_results": results},
                  f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 60)
    print("[3/3] Evaluation complete. Summary:")
    print(f"      Total queries         : {summary['total_queries']}")
    print(f"      Mean Recall@{RECALL_K}        : {avg_recall}")
    print(f"      Intent Accuracy       : {intent_accuracy}")
    print(f"      Generation Faithful   : {generation_acc}")
    print(f"      Mean Latency          : {avg_latency}s")
    print("\n  Results saved -> " + OUTPUT_FILE)
    print("=" * 60)

run_evaluation()

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  South Asian Culinary RAG — Benchmark Evaluation

[1/3] Verifying LangGraph pipeline in memory...
      Pipeline detected in memory.

[2/3] Running queries...

  [q001] How do I make pizza?
--- CLASSIFIER: NON_SOUTH_ASIAN ---
         intent=NON_SOUTH_ASIAN | recall=None | faithful=None | latency=0.002s
  [q002] yes
--- CLASSIFIER: ALTERNATIVE_REQUEST ---
--- RETRIEVING FROM FAISS: What is a good South Asian alternative to how do i make pizza?? ---
=========DOCS========= [Document(id='eac9fe5a-21d8-4b3c-beb3-fcb1fbf012b1', metadata={'title': 'Shirmal (Persian Saffron Flatbread)', 'source_url': 'https://en.wikibooks.org/wiki/Cookbook:Shirmal_(Persian_Saffron_Flatbread)', 'cuisine_type': 'South Asian', 'diet': 'veg', 'prep_time': 'slow', 'dish_type': 'dry-main', 'db_chunk_id': 'wiki_southasian_151_instructions', 'content_type': 'instructions'}, page_content='Dish: Shirmal (Persian Saffron Flatbread)\nSection: Instructions\n\n1. Soak the saffron strands in the warm milk. Set aside.\n2. S

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


         intent=ALTERNATIVE_REQUEST | recall=None | faithful=None | latency=6.675s
  [q003] surprise me
--- CLASSIFIER: VAGUE_REQUEST ---
         intent=VAGUE_REQUEST | recall=None | faithful=None | latency=0.001s
  [q004] rice-based, vegetarian, quick
--- CLASSIFIER: INGREDIENTS_ONLY ---
         intent=INGREDIENTS_ONLY | recall=None | faithful=None | latency=0.0s
  [q005] rice, lentils, turmeric
--- CLASSIFIER: INGREDIENTS_ONLY ---
         intent=INGREDIENTS_ONLY | recall=None | faithful=None | latency=0.0s
  [q006] a spicy curry
--- CLASSIFIER: DISH_QUERY ---
--- RETRIEVING FROM FAISS: a spicy curry rice lentils ---
=========DOCS========= [Document(id='80a90362-c134-4dda-b7d6-939f844defb8', metadata={'title': 'Bhuna Khichuri (Bengali Rice and Lentils)', 'source_url': 'https://en.wikibooks.org/wiki/Cookbook:Bhuna_Khichuri_(Bengali_Rice_and_Lentils)', 'cuisine_type': 'South Asian', 'diet': 'veg', 'prep_time': 'quick', 'dish_type': 'rice', 'db_chunk_id': 'wiki_southasian_008_instruct

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


         intent=RECIPE_REQUEST | recall=1.0 | faithful=True | latency=5.401s
  [q071] Thalassery Biryani recipe
--- CLASSIFIER: DISH_QUERY ---
--- RETRIEVING FROM FAISS: Thalassery Biryani recipe ---
=========DOCS========= [Document(id='a6f39bc8-530b-4365-84cc-e5b7d7b19ead', metadata={'title': 'Hyderabad Biryani', 'source_url': 'https://en.wikibooks.org/wiki/Cookbook:Hyderabad_Biryani', 'cuisine_type': 'South Asian', 'diet': 'non-veg', 'prep_time': 'slow', 'dish_type': 'rice', 'db_chunk_id': 'wiki_southasian_054_instructions', 'content_type': 'instructions'}, page_content="Dish: Hyderabad Biryani\nSection: Instructions\n\n1. Make deep incisions in the chicken flesh. They should be deep enough for spices to get absorbed, but not so deep that they could render the pieces smaller.\n2. Mix turmeric, chilli powder, salt, garlic paste, yogurt, and half-lemon's juice. Thoroughly apply this paste onto the chicken, and let marinate for an hour.\n3. Heat about 100 ml of oil in a pan. Add cumin, 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=========DOCS========= [Document(id='4b4ccc7e-9d4b-4930-9460-367a16fa6592', metadata={'title': 'Dharwad Pedha (Sweetened Paneer Cheese)', 'source_url': 'https://en.wikibooks.org/wiki/Cookbook:Dharwad_Pedha_(Sweetened_Paneer_Cheese)', 'cuisine_type': 'South Asian', 'diet': 'veg', 'prep_time': 'slow', 'dish_type': 'dry-main', 'db_chunk_id': 'wiki_southasian_043_ingredients', 'content_type': 'ingredients'}, page_content='Dish: Dharwad Pedha (Sweetened Paneer Cheese)\nSection: Ingredients\n\n- Milk (full fat)\n- Lime juice\n- Ghee\n- ½ cup milk\n- ½ cup white granulated sugar\n- Cardamom powder\n- Sugar for garnish'), Document(id='758644d3-299d-4f3c-b46d-30f6babc17e9', metadata={'title': 'Dharwad Pedha (Sweetened Paneer Cheese)', 'source_url': 'https://en.wikibooks.org/wiki/Cookbook:Dharwad_Pedha_(Sweetened_Paneer_Cheese)', 'cuisine_type': 'South Asian', 'diet': 'veg', 'prep_time': 'slow', 'dish_type': 'dry-main', 'db_chunk_id': 'wiki_southasian_043_instructions', 'content_type': 'instruct

KeyboardInterrupt: 